In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==========================================================
# PARAMETERS
# ==========================================================

DB = "customer_health"
BRONZE = "BRONZE"
SILVER = "SILVER"

spark.conf.set("spark.sql.session.timeZone", "UTC")

try:

    # ==========================================================
    # CREATE TRANSIENT TABLE
    # ==========================================================

    spark.createDataFrame(
        [],
        """
        raw_data string,
        insert_date timestamp
        """
    ).write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(
        f"{DB}.{SILVER}.CUSTOM_ACTIVITIES_TRANSIENT"
     )

    # ==========================================================
    # LOAD TRANSIENT TABLE
    # ==========================================================

    (
        spark.table(
            f"{DB}.{BRONZE}.CUSTOM_ACTIVITES_RAW"
        )
        .select(
            "raw_data",
            "insert_date"
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{DB}.{SILVER}.CUSTOM_ACTIVITIES_TRANSIENT"
        )
    )

    transient_df = spark.table(
        f"{DB}.{SILVER}.CUSTOM_ACTIVITIES_TRANSIENT"
    )

    # ==========================================================
    # CREATE TARGET TABLE IF NOT EXISTS
    # ==========================================================

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB}.{SILVER}.CUSTOM_ACTIVITIES
    (
        custom_activity_id STRING,
        custom_activity_name STRING,
        custom_activity_outcome_id STRING,
        custom_activity_outcome_name STRING,
        md5_hash STRING,
        insert_date TIMESTAMP,
        update_date TIMESTAMP
    )
    USING DELTA
    """)

    # ==========================================================
    # PARSE JSON USING UDF
    # ==========================================================

    # Use the parse_activity_final UDF to extract and clean the JSON
    parsed_df = (
        transient_df
        .withColumn(
            "parsed_json",
            F.expr("customer_health.SILVER.PARSE_ACTIVITY_FINAL(raw_data)")
        )
        .withColumn(
            "activity_array",
            F.from_json(
                F.expr("get_json_object(parsed_json, '$.data')"),
                """
                array<
                    struct<
                        id:string,
                        name:string,
                        fields:array<
                            struct<
                                id:string,
                                name:string
                            >
                        >
                    >
                >
                """
            )
        )
    )

    # ==========================================================
    # FLATTEN ACTIVITIES
    # ==========================================================

    print("Checking activity_array before explode:")
    display(parsed_df.select("activity_array").limit(3))

    activities_df = (
        parsed_df
        .withColumn(
            "activity",
            F.explode("activity_array")
        )
    )

    print(f"activities_df count: {activities_df.count()}")
    display(activities_df.select("activity").limit(2))

    # ==========================================================
    # FLATTEN FIELDS
    # ==========================================================

    custom_df = (
        activities_df
        .withColumn(
            "field",
            F.explode("activity.fields")
        )
        .select(
            F.col("activity.id").alias(
                "custom_activity_id"
            ),
            F.col("activity.name").alias(
                "custom_activity_name"
            ),
            F.col("field.id").alias(
                "custom_activity_outcome_id"
            ),
            F.col("field.name").alias(
                "custom_activity_outcome_name"
            )
        )
        .distinct()
    )

    # ==========================================================
    # MD5 HASH
    # ==========================================================

    custom_df = (
        custom_df
        .withColumn(
            "md5_hash",
            F.md5(
                F.concat_ws(
                    "||",
                    F.coalesce(
                        F.col("custom_activity_name"),
                        F.lit("")
                    ),
                    F.coalesce(
                        F.col("custom_activity_outcome_name"),
                        F.lit("")
                    )
                )
            )
        )
        .withColumn(
            "insert_date",
            F.current_timestamp()
        )
        .withColumn(
            "update_date",
            F.current_timestamp()
        )
    )

    print(f"custom_df count: {custom_df.count()}")
    display(custom_df.limit(5))

    # ==========================================================
    # MERGE TARGET
    # ==========================================================

    target = DeltaTable.forName(
        spark,
        f"{DB}.{SILVER}.CUSTOM_ACTIVITIES"
    )

    update_map = {
        "custom_activity_name":
            "src.custom_activity_name",
        "custom_activity_outcome_name":
            "src.custom_activity_outcome_name",
        "md5_hash":
            "src.md5_hash",
        "update_date":
            "current_timestamp()"
    }

    insert_map = {
        "custom_activity_id":
            "src.custom_activity_id",
        "custom_activity_name":
            "src.custom_activity_name",
        "custom_activity_outcome_id":
            "src.custom_activity_outcome_id",
        "custom_activity_outcome_name":
            "src.custom_activity_outcome_name",
        "md5_hash":
            "src.md5_hash",
        "insert_date":
            "current_timestamp()",
        "update_date":
            "current_timestamp()"
    }

    (
        target.alias("tgt")
        .merge(
            custom_df.alias("src"),
            """
            tgt.custom_activity_id =
                src.custom_activity_id
            AND
            tgt.custom_activity_outcome_id =
                src.custom_activity_outcome_id
            """
        )
        .whenMatchedUpdate(
            condition="""
            tgt.md5_hash <> src.md5_hash
            """,
            set=update_map
        )
        .whenNotMatchedInsert(
            values=insert_map
        )
        .execute()
    )

    print(
        "Store Procedure Executed Successfully"
    )

except Exception as e:

    raise Exception(
        f"Error: {str(e)}"
    )